# 03 — Match Plan Codes

There's no reliable automatic way to map a vBill `PlanCode` to a OneBill
product/price-plan — a bare code like `SC-04047` doesn't tell you (or the
code) what service it represents unless the two catalogs happen to share
identical codes, which they mostly don't. So this step is **manual by
design**: it builds a template listing every distinct vBill plan code in
use, plus a reference copy of the real OneBill catalog to pick from, and
you fill in the OneBill `product_name` / `priceplan_name` for each row.

**Flow**

1. Pull every distinct `PlanCode` from the subscriptions table (with usage
   counts, so you can prioritize the ones actually worth mapping first).
2. Pre-fill any rows where the vBill code happens to exactly match a
   OneBill `priceplan_code` (a bonus, not the main path).
3. Save the template + a reference copy of the OneBill catalog.
4. **You edit `03_plan_code_mapping_template.csv` by hand** — for each
   `PlanCode`, look it up in `03_onebill_catalog_reference.csv` (or
   wherever else you know what the plan actually is) and fill in
   `product_name` / `priceplan_name`. Leave a row blank if you don't know
   the mapping yet — unmapped codes fall back to `STATIC_FALLBACK_PLAN` in
   `07_Create_Subscription_Orders.ipynb`, they don't block the import.
5. Re-run from **"Finalize"** onward — validates what you typed against
   the real catalog (catches typos before they become a rejected order)
   and saves `03_plan_code_mapping.csv`, the file `07_Create_Subscription_Orders.ipynb`
   actually reads.

> **TODO**: confirm the plan-code column name — see
> `SUBSCRIPTION_PLANCODE_COLUMN` in `onebill_common.py` (currently
> `"PlanCode"`). If your subscriptions table has a plan name/description
> column too, add it to `SUBSCRIPTION_PLAN_CONTEXT_COLUMNS` in
> `onebill_common.py` — it'll get pulled into the template alongside the
> code to give you something to recognize the plan by.

## 1. Setup

In [9]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from sqlalchemy import create_engine

logger = get_logger("match_plan_codes")

df_available_priceplans = load_df("products_available")
logger.info(f"Loaded {len(df_available_priceplans):,} available price-plan rows from 02_Fetch_Products.ipynb")


2026-07-28 10:31:42,758 [INFO] Loaded 1 available price-plan rows from 02_Fetch_Products.ipynb


## 2. Distinct vBill plan codes actually in use

Includes a usage count (`subscription_count`) and any extra context
columns configured in `SUBSCRIPTION_PLAN_CONTEXT_COLUMNS`, so you can
prioritize which codes are worth mapping first and have something more
than a bare code to recognize each plan by.

In [10]:
assert BI_DATASTORE_URL, "DB_USERNAME/DB_PASSWORD/DB_HOST not set in .env"
engine = create_engine(BI_DATASTORE_URL)

context_cols_sql = "".join(f", MAX(`{col}`) AS `{col}`" for col in SUBSCRIPTION_PLAN_CONTEXT_COLUMNS)

PLANCODE_USAGE_QUERY = f'''
SELECT 
    PlanCode
    ,COUNT(*) AS subscription_count
FROM bi_datastore.billing_subscription
WHERE _DataSource = 'vBill'
AND AccountCode = '99965692'
AND PlanCode IS NOT NULL AND PlanCode != ''
GROUP BY PlanCode;
'''.strip()

df_plan_codes = pd.read_sql(PLANCODE_USAGE_QUERY, con=engine)
logger.info(f"{len(df_plan_codes):,} distinct vBill plan codes in use "
            f"(covering {df_plan_codes['subscription_count'].sum():,} subscriptions)")
df_plan_codes.head(20)


2026-07-28 10:31:53,763 [INFO] 12 distinct vBill plan codes in use (covering 323 subscriptions)


,PlanCode,subscription_count
0,5defa0dd,1
1,9b947601,1
2,SC-02132,6
3,SC-02133,2
4,SC-02134,2
5,SC-02138,2
6,SC-02139,3
7,SC-02140,12
8,SC-04045,194
9,SC-04046,93


## 3. Pre-fill exact code matches (bonus, not the main path)

Only fires when a vBill `PlanCode` happens to exactly equal a OneBill
`priceplan_code` — worth doing since it's free, but expect most rows to
stay blank and need a manual fill-in.

In [11]:
def _norm(code):
    return str(code).strip().upper() if code is not None else None

df_available_priceplans["_match_key"] = df_available_priceplans["priceplan_code"].map(_norm)
df_plan_codes["_match_key"] = df_plan_codes["PlanCode"].map(_norm)

prefill = df_available_priceplans.drop_duplicates(subset=["_match_key"])[
    ["_match_key", "product_name", "priceplan_name"]
]

df_template = df_plan_codes.merge(prefill, on="_match_key", how="left").drop(columns=["_match_key"])
df_template["product_name"] = df_template["product_name"].fillna("")     # <-- fill in the rest by hand
df_template["priceplan_name"] = df_template["priceplan_name"].fillna("")  # <-- fill in the rest by hand

exact_matches = (df_template["product_name"] != "").sum()
logger.info(f"{exact_matches:,}/{len(df_template):,} plan codes pre-filled by an exact code match; "
            f"{len(df_template) - exact_matches:,} need a manual look-up")

df_template.head(20)


2026-07-28 10:32:15,612 [INFO] 0/12 plan codes pre-filled by an exact code match; 12 need a manual look-up


,PlanCode,subscription_count,product_name,priceplan_name
0,5defa0dd,1,,
1,9b947601,1,,
2,SC-02132,6,,
3,SC-02133,2,,
4,SC-02134,2,,
5,SC-02138,2,,
6,SC-02139,3,,
7,SC-02140,12,,
8,SC-04045,194,,
9,SC-04046,93,,


## 4. Save the template + a reference copy of the OneBill catalog

**Stop here and edit `03_plan_code_mapping_template.csv` by hand** — for
every row with blank `product_name`/`priceplan_name`, look the `PlanCode`
up (using `subscription_count` and any context columns to help identify
it) and copy the matching `product_name` + `priceplan_name` **exactly** as
they appear in `03_onebill_catalog_reference.csv` — the finalize step
below checks for an exact match and will flag anything that doesn't line
up. Save the file, then come back and run from "Finalize" onward.

In [4]:
save_df("plan_mapping_template", df_template)
save_df("plan_mapping_reference", df_available_priceplans.drop(columns=["_match_key"]))
print("\nNow edit migration_data/03_plan_code_mapping_template.csv by hand, then re-run from 'Finalize' below.")


Saved 12 rows -> migration_data\03_plan_code_mapping_template.csv
Saved 1 rows -> migration_data\03_onebill_catalog_reference.csv

Now edit migration_data/03_plan_code_mapping_template.csv by hand, then re-run from 'Finalize' below.


## 5. Finalize — validate your edits and save the mapping `07_Create_Subscription_Orders.ipynb` reads

In [7]:
df_template_edited = load_df("plan_mapping_template")
df_catalog = load_df("plan_mapping_reference")

df_validated = validate_plan_mapping(df_template_edited, df_catalog)

mismatches = df_validated[df_validated["catalog_match"] == False]  # noqa: E712
blanks     = df_validated[df_validated["catalog_match"] == "blank"]
matched    = df_validated[df_validated["catalog_match"] == True]  # noqa: E712

logger.info(f"{len(matched):,} rows match the OneBill catalog exactly, "
            f"{len(blanks):,} left blank (will use STATIC_FALLBACK_PLAN), "
            f"{len(mismatches):,} DO NOT match the catalog — fix these before proceeding")

mismatches


2026-07-28 10:28:51,709 [INFO] 0 rows match the OneBill catalog exactly, 3 left blank (will use STATIC_FALLBACK_PLAN), 9 DO NOT match the catalog — fix these before proceeding


,PlanCode,subscription_count,product_name,priceplan_name,catalog_match
2,SC-02132,6,Wholesale Fibre BS2 (Enable),Fibre 920,False
3,SC-02133,2,Wholesale Fibre BS2 (Enable),Fibre 500,False
4,SC-02134,2,Wholesale Fibre BS2 (Enable),Fibre 100,False
5,SC-02138,2,Wholesale Fibre BS2 (Chorus),Fibre 920,False
6,SC-02139,3,Wholesale Fibre BS2 (Chorus),Fibre 500,False
7,SC-02140,12,Wholesale Fibre BS2 (Chorus),Fibre 100,False
8,SC-04045,194,Wholesale Fibre BS2 (Enable),Fibre 100,False
9,SC-04046,93,Wholesale Fibre BS2 (Chorus),Fibre 100,False
10,SC-04047,6,Wholesale Fibre BS2 (TFF),Fibre 100,False


`mismatches` above means the `product_name`/`priceplan_name` you typed
don't exist together in `03_onebill_catalog_reference.csv` — almost always
a typo or a name that's changed since the reference was pulled. Fix those
rows in the template CSV and re-run this cell (and the one above it)
before saving.

In [12]:
df_template_edited = load_df("plan_mapping_template")
save_df("plan_mapping", df_template_edited)

Saved 12 rows -> migration_data\03_plan_code_mapping.csv


In [8]:
assert mismatches.empty, (
    f"{len(mismatches)} row(s) don't match the OneBill catalog — fix them in "
    f"03_plan_code_mapping_template.csv (see 'mismatches' above) before saving."
)

df_final_mapping = matched[["PlanCode", "product_name", "priceplan_name"]].reset_index(drop=True)
save_df("plan_mapping", df_final_mapping)


AssertionError: 9 row(s) don't match the OneBill catalog — fix them in 03_plan_code_mapping_template.csv (see 'mismatches' above) before saving.